In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv("../data/feature_engineered_dataset.csv")
df["CourseCategory"].value_counts()

CourseCategory
Programming                5
Design                     5
Business                   5
Marketing                  5
Data Science               5
Machine Learning           5
Cybersecurity              5
Project Management         5
Finance                    5
Artificial Intelligence    5
Web Development            5
Digital Marketing          5
Name: count, dtype: int64

In [3]:
Xe_train, Xe_test, ye_train, ye_test = joblib.load(
    "../models/enrollment_data.pkl"
)

Xr_train, Xr_test, yr_train, yr_test = joblib.load(
    "../models/revenue_data.pkl"
)

preprocessor = joblib.load(
    "../models/preprocessor.pkl"
)

In [4]:
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    "GradientBoosting": GradientBoostingRegressor(
        random_state=42
    )
}

In [5]:
def evaluate_model(
    model,
    X_train,
    X_test,
    y_train,
    y_test
):

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    return {
        "Pipeline": pipeline,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [6]:
enrollment_results = []

for name, model in models.items():

    result = evaluate_model(
        model,
        Xe_train,
        Xe_test,
        ye_train,
        ye_test
    )

    enrollment_results.append({
        "Model": name,
        "MAE": result["MAE"],
        "RMSE": result["RMSE"],
        "R2": result["R2"]
    })

enrollment_df = pd.DataFrame(
    enrollment_results
)

enrollment_df.sort_values(
    "R2",
    ascending=False
)

,Model,MAE,RMSE,R2
3,RandomForest,10.320417,12.321273,0.166876
1,Ridge,10.536660,13.178785,0.046876
2,Lasso,10.542909,13.268801,0.033811
4,GradientBoosting,12.057133,15.678104,-0.348919
0,Linear,22.206082,27.308826,-3.092651


In [7]:
revenue_results = []

for name, model in models.items():

    result = evaluate_model(
        model,
        Xr_train,
        Xr_test,
        yr_train,
        yr_test
    )

    revenue_results.append({
        "Model": name,
        "MAE": result["MAE"],
        "RMSE": result["RMSE"],
        "R2": result["R2"]
    })

revenue_df = pd.DataFrame(
    revenue_results
)

revenue_df.sort_values(
    "R2",
    ascending=False
)

,Model,MAE,RMSE,R2
1,Ridge,3288.680487,4607.535681,0.972649
4,GradientBoosting,3267.531197,4858.486979,0.969589
3,RandomForest,3292.436825,5220.195205,0.964892
2,Lasso,5006.718827,6380.276263,0.947554
0,Linear,5031.111779,6406.582303,0.947121


In [8]:
# (Feature importance is computed in 07_feature_importance.ipynb)

In [9]:
# Removed: category modelling on per-course rows.
# The category-level model lives in 05b_category_model.ipynb.

In [10]:
best_revenue_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=200,
            random_state=42
        )
    )
])

best_revenue_pipeline.fit(
    Xr_train,
    yr_train
)

joblib.dump(
    best_revenue_pipeline,
    "../models/revenue_model.pkl"
)

print("Revenue model saved")

Revenue model saved


In [11]:
# (Feature importance is computed in 07_feature_importance.ipynb)

In [12]:
print(revenue_df)

              Model          MAE         RMSE        R2
0            Linear  5031.111779  6406.582303  0.947121
1             Ridge  3288.680487  4607.535681  0.972649
2             Lasso  5006.718827  6380.276263  0.947554
3      RandomForest  3292.436825  5220.195205  0.964892
4  GradientBoosting  3267.531197  4858.486979  0.969589


In [13]:
print("Enrollment models:")
print(enrollment_df.sort_values("R2", ascending=False))

print("\nRevenue models:")
print(revenue_df.sort_values("R2", ascending=False))

Enrollment models:
              Model        MAE       RMSE        R2
3      RandomForest  10.320417  12.321273  0.166876
1             Ridge  10.536660  13.178785  0.046876
2             Lasso  10.542909  13.268801  0.033811
4  GradientBoosting  12.057133  15.678104 -0.348919
0            Linear  22.206082  27.308826 -3.092651

Revenue models:
              Model          MAE         RMSE        R2
1             Ridge  3288.680487  4607.535681  0.972649
4  GradientBoosting  3267.531197  4858.486979  0.969589
3      RandomForest  3292.436825  5220.195205  0.964892
2             Lasso  5006.718827  6380.276263  0.947554
0            Linear  5031.111779  6406.582303  0.947121


In [14]:
# 1. Build and train the Best Enrollment Pipeline (Random Forest)
best_enrollment_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])
best_enrollment_pipeline.fit(Xe_train, ye_train)

# 2. Build and train the Best Revenue Pipeline (Ridge Regression)
best_revenue_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])
best_revenue_pipeline.fit(Xr_train, yr_train)

# 3. Export both completed pipelines to your models directory
joblib.dump(best_enrollment_pipeline, "../models/enrollment_model.pkl")
joblib.dump(best_revenue_pipeline, "../models/revenue_model.pkl")

print("Both pipelines successfully trained and saved!")

Both pipelines successfully trained and saved!


In [15]:
df.to_csv("../data/model_data.csv", index=False)

In [16]:
# Print the engineered features used for the Revenue Model
print("Revenue Columns:")
print(Xr_train.columns.tolist())

print("\nEnrollment Columns:")
# Print the engineered features used for the Enrollment Model
print(Xe_train.columns.tolist())

Revenue Columns:
['CourseCategory', 'CourseType', 'CourseLevel', 'CoursePrice', 'CourseDuration', 'CourseRating', 'Expertise', 'YearsOfExperience', 'PriceBand', 'DurationBucket', 'RatingTier', 'ExperienceBucket', 'ExpertiseMatch', 'LevelEncoded']

Enrollment Columns:
['CourseCategory', 'CourseType', 'CourseLevel', 'CoursePrice', 'CourseDuration', 'CourseRating', 'Expertise', 'YearsOfExperience', 'PriceBand', 'DurationBucket', 'RatingTier', 'ExperienceBucket', 'ExpertiseMatch', 'LevelEncoded']


In [17]:
import joblib

model = joblib.load("../models/revenue_model.pkl")

print(type(model))

<class 'sklearn.pipeline.Pipeline'>


In [19]:
# Convert the list of results into a pandas DataFrame first
enrollment_results_df = pd.DataFrame(enrollment_results)

# Now save the DataFrame cleanly to a CSV file
enrollment_results_df.to_csv(
    "../data/enrollment_results.csv",
    index=False
)

print("Enrollment results successfully converted and saved!")

Enrollment results successfully converted and saved!


In [20]:
# Convert the revenue results list into a pandas DataFrame first
revenue_results_df = pd.DataFrame(revenue_results)

# Now save the DataFrame cleanly to your data directory
revenue_results_df.to_csv(
    "../data/revenue_results.csv",
    index=False
)

print("Revenue results successfully converted and saved!")

Revenue results successfully converted and saved!
